<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Claude API for Python Developers</h1>
<h1>2. Claude Models</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import anthropic
from IPython.display import display, Markdown, Image

import matplotlib.pyplot as plt

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.3
IPython version      : 9.8.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.1.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 87485bac20625f43da5809d38ec119aca4321f71

IPython   : 9.8.0
watermark : 2.5.0
anthropic : 0.75.0
matplotlib: 3.10.7



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Basic Setup

The first step is generate API key on the Claude website and store it as the "`ANTHROPIC_API_KEY`" variable in your local environment. Without it we won't be able to do anything. You can find your API key in your using settings: https://www.merge.dev/blog/anthropic-api-key

In [4]:
client = anthropic.Anthropic()

We start by getting the list of available models:

In [5]:
model_list = client.models.list()

In [6]:
print("\n".join(model.id for model in model_list.data))

claude-opus-4-5-20251101
claude-haiku-4-5-20251001
claude-sonnet-4-5-20250929
claude-opus-4-1-20250805
claude-opus-4-20250514
claude-sonnet-4-20250514
claude-3-7-sonnet-20250219
claude-3-5-haiku-20241022
claude-3-haiku-20240307
claude-3-opus-20240229


A Basic example: Text generation with Claude

In [7]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Complete this sentence in a creative way: 'The robot opened its eyes and saw...'"
        }
    ]
)

After the API call we get back a response object:

In [8]:
response

Message(id='msg_01W2JKu2T7zkakUQXhMwVUFR', content=[TextBlock(citations=None, text='The robot opened its eyes and saw a world painted in frequencies it had never been programmed to perceive—where emotions hung in the air like visible aurora, where the laughter of children created spiraling golden threads, and where its own reflection in a nearby window revealed not circuits and steel, but something that looked remarkably like wonder.', type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=26, output_tokens=71, server_tool_use=None, service_tier='standard'))

which contains the output information in the content

In [9]:
response.content[0].text

'The robot opened its eyes and saw a world painted in frequencies it had never been programmed to perceive—where emotions hung in the air like visible aurora, where the laughter of children created spiraling golden threads, and where its own reflection in a nearby window revealed not circuits and steel, but something that looked remarkably like wonder.'

We also declare a helper function to help us display our output:

In [10]:
def show_response(response):
    """Display Claude's response with markdown formatting."""
    display(Markdown(response.content[0].text))

In [11]:
show_response(response)

The robot opened its eyes and saw a world painted in frequencies it had never been programmed to perceive—where emotions hung in the air like visible aurora, where the laughter of children created spiraling golden threads, and where its own reflection in a nearby window revealed not circuits and steel, but something that looked remarkably like wonder.

---
## Basic Usage

Claude offers different models optimized for various use cases:

| Model | ID | Best For |
|-------|-----|----------|
| Claude 4.5 Sonnet | `claude-sonnet-4-5-20250929` | Best balance of speed and intelligence |
| Claude 4.5 Opus | `claude-opus-4-5-20251101` | Most capable for complex tasks |
| Claude 4.5 Haiku | `claude-haiku-4-5-20251001` | Fast, efficient for high volume |


The simplest possible API call

In [12]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "What is machine learning in one sentence?"}
    ]
)

In [13]:
show_response(response)

Machine learning is a type of artificial intelligence where computers learn to make predictions or decisions by finding patterns in data, rather than being explicitly programmed with rules.

## Attention

The model pays attention to relevant context and connects different parts of the prompt

In [14]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": """Read this passage and answer the question:

Passage: Sarah went to the store. She bought apples, bread, and milk. 
On her way home, she met her friend Tom. Tom was carrying a large box.
He said he had just bought a new computer. Sarah congratulated him.

Question: Who bought a new computer?"""
        }
    ]
)

show_response(response)

According to the passage, Tom bought a new computer. The text states that when Sarah met Tom on her way home, "He said he had just bought a new computer."

## In context learning

The model learns the pattern from examples without explicit training

In [15]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": """Convert these sentences to pig latin:

Input: hello -> Output: ellohay
Input: world -> Output: orldway
Input: python -> Output: ythonpay
Input: claude -> Output:"""
        }
    ]
)

In [16]:
show_response(response)

Looking at the pattern in your examples, I can see the pig latin rule being applied:

- For words starting with consonants: move the initial consonant(s) to the end and add "ay"

Following this pattern:

Input: claude -> Output: **laudecay**

(The initial consonant "c" moves to the end, then "ay" is added: c-laude → laude-c → laudec-ay)

## Temperature

In [17]:
prompt = "Write a one-sentence story about a cat."
temperatures = [0.0, 0.5, 1.0]

print(f"Prompt: {prompt}\n")
print("=" * 60)

for temp in temperatures:
    print(f"\nTemperature = {temp}")
    print("-" * 40)
    
    # Generate 3 responses at each temperature
    for i in range(3):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=100,
            temperature=temp,
            messages=[{"role": "user", "content": prompt}]
        )
        print(f"  {i+1}. {response.content[0].text}")

Prompt: Write a one-sentence story about a cat.


Temperature = 0.0
----------------------------------------
  1. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this would be the last warm day before her family moved away and left her behind.
  2. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this would be the last warm day before her family moved away and left her behind.
  3. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this would be the last warm day before her family moved away and left her behind.

Temperature = 0.5
----------------------------------------
  1. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this would be the last warm day before her family moved away and left her behind.
  2. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this was the last warm day before her family moved away forever.
  3. The old tabby cat stretched lazily in 

## System Prompts

System prompts set the context, personality, and constraints for Claude's responses.

### Technical Expert

In [18]:
response_expert = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    system="You are a senior Python developer. Explain concepts with code examples when relevant.",
    messages=[
        {"role": "user", "content": "What are decorators?"}
    ]
)

print("🔧 Technical Expert Response:")
show_response(response_expert)

🔧 Technical Expert Response:


Decorators are a powerful Python feature that allows you to modify or extend the behavior of functions, methods, or classes without permanently changing their code. Think of them as "wrappers" that add functionality to existing code.

## Basic Concept

A decorator is essentially a function that takes another function as an argument and returns a modified version of that function.

```python
def my_decorator(func):
    def wrapper():
        print("Something before the function")
        func()
        print("Something after the function")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
# Output:
# Something before the function
# Hello!
# Something after the function
```

The `@my_decorator` syntax is equivalent to:
```python
say_hello = my_decorator(say_hello)
```

## Common Use Cases

### 1

### Friendly teacher

In [19]:
response_teacher = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    system="You are a friendly teacher explaining to a beginner. Use simple language and analogies.",
    messages=[
        {
            "role": "user", 
             "content": "What are decorators?"
        }
    ]
)

print("📚 Friendly Teacher Response:")
show_response(response_teacher)

📚 Friendly Teacher Response:


Great question! Let me explain decorators in a simple way.

## What are decorators?

Think of a decorator like **gift wrapping** 🎁. You have a present (your original function), and you want to add some extra features to it without changing what's inside the box. The decorator is like the wrapping paper that adds something special on the outside.

## A Simple Example

Let's say you have a function that greets someone:

```python
def say_hello(name):
    return f"Hello, {name}!"
```

Now, imagine you want to make this greeting more fancy by adding some decorative stars around it. Instead of changing the original function, you can use a decorator:

```python
def add_stars(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"⭐ {result} ⭐"
    return wrapper

@add_stars
def say_hello(name):
    return f"Hello, {name}!"

print(say_hello("Alice"))
# Output: ⭐ Hello, Alice! ⭐
```

## How it works (step by step):

1. The `@add_stars` is like saying "wrap this function with the add_stars decorator"
2. When you call `say_hello("Alice")`, it actually calls the wrapper function
3. The wrapper function calls your original `say_hello` function
4. Then it adds the stars around the result

## Real-world analogy:

It's like having a **personal assistant** for your function:
- Your function does its main job (like greeting people)
- The decorator assistant adds extra services (like adding fancy formatting, logging when the function runs, checking permissions, etc.)
- The person calling your function doesn't need to know about the assistant - they just see the enhanced result!

## Common uses:
- Adding logging ("This function was called at 2:30 PM")
- Checking if a user has permission to do something
- Measuring how long a function takes to run
- Adding retry logic if something fails

Does this help clarify what decorators are? Would you like me to explain any part in more detail?

### Assistant Prefilling

You can start Claude's response with specific text to guide the format or direction.


In [20]:
# Prefilling to set tone/style
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[
        {"role": "user", "content": "Explain recursion."},
        # Forces analogy-style response
        {"role": "assistant", "content": "Recursion is like"}  
    ]
)

In [21]:
show_response(response)

 a nested doll that contains smaller versions of itself. It's a programming concept where a function calls itself to solve a problem by breaking it down into smaller, similar subproblems.

## Key Components

**1. Base Case**
- A condition that stops the recursion
- Without it, the function would call itself forever
- The "smallest" version of the problem that can be solved directly

**2. Recursive Case**
- The function calls itself with a modified input
- Each call should move closer to the base case

## Simple Example: Factorial

```python
def factorial(n):
    # Base case
    if n <= 1:
        return 1
    
    # Recursive case
    return n * factorial(n - 1)

# factorial(5) = 5 * factorial(4) = 5 * 24 = 120
```

## How It Works (Step by Step)

```
factorial(5)
├── 5 * factorial(4)
    ├── 4 * factorial(3)
        ├── 3 * factorial(2)
            ├── 2 * factorial(1)
                └── returns 1 (base case)
            └── returns 2 * 1 = 2
        └── returns 3 * 2

##  Input Formatting

Proper formatting helps Claude understand your intent and produce better responses.

### XML Tags

Claude was trained to understand XML-style tags for structuring prompts.

In [22]:
prompt = """
<document>
Python is a high-level programming language known for its clear syntax and readability.
It was created by Guido van Rossum and first released in 1991.
Python supports multiple programming paradigms including procedural, object-oriented, 
and functional programming.
</document>

<task>
Extract the following information from the document:
1. Creator's name
2. Year of first release
3. Programming paradigms supported
</task>

<output_format>
Respond in a structured format with clear labels.
</output_format>
"""

In [23]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[{"role": "user", "content": prompt}]
)

In [24]:
show_response(response)

Based on the document, here is the extracted information:

**Creator's name:** Guido van Rossum

**Year of first release:** 1991

**Programming paradigms supported:** 
- Procedural programming
- Object-oriented programming
- Functional programming

### Markdown Formatting

Claude handles markdown well for structured inputs.


In [25]:
# Using markdown formatting
prompt = """
# Task: Code Review

Please review the following Python function:

```python
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total = total + n
    average = total / len(numbers)
    return average
```

## Review Criteria:
- Code correctness
- Error handling
- Pythonic style
- Performance

Please provide specific suggestions for improvement.
"""

In [26]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=400,
    messages=[{"role": "user", "content": prompt}]
)

In [27]:
print("Code review response:")
show_response(response)

Code review response:


# Code Review: `calculate_average` Function

## Overall Assessment
The function works correctly for the happy path but has several areas for improvement, particularly around error handling and Pythonic style.

## Issues and Suggestions

### 1. **Error Handling** ⚠️ **Critical**
```python
# Current code fails with ZeroDivisionError
calculate_average([])  # Raises ZeroDivisionError

# Also fails with TypeError for non-numeric inputs
calculate_average(['a', 'b'])  # Raises TypeError during addition
```

**Suggestions:**
```python
def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate average of empty sequence")
    
    # Input validation
    if not all(isinstance(x, (int, float)) for x in numbers):
        raise TypeError("All elements must be numeric")
    
    return sum(numbers) / len(numbers)
```

### 2. **Pythonic Style** 📝 **Recommended**
- Use built-in `sum()` function instead of manual loop
- More concise and readable

```python
# Instead of:
total = 0
for n in numbers:
    total = total + n

# Use:
total = sum(numbers)
```

### 3. **Performance** ⚡ **Minor**
- Built-in `sum()` is implemented in C and faster than Python loops
- For large datasets, the difference becomes noticeable

### 4. **Type Hints** 📋 **Best Practice**
```python
from typing import Sequence, Union

def calculate_average(numbers: Sequence[Union[int, float]]) -> float:
    # function body
```

##

### Few-Shot Examples

Providing examples in your prompt helps Claude understand the desired format.


In [28]:
# Few-shot prompting for consistent output format
prompt = """
Classify the sentiment of the following texts.

<examples>
Text: "I love this product! Best purchase ever!"
Sentiment: POSITIVE
Confidence: HIGH

Text: "The delivery was okay, nothing special."
Sentiment: NEUTRAL
Confidence: MEDIUM

Text: "Terrible experience, never buying again."
Sentiment: NEGATIVE
Confidence: HIGH
</examples>

Now classify this text:
Text: "The food was delicious but the service was slow."
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[{"role": "user", "content": prompt}]
)

print("Few-shot classification:")
print(response.content[0].text)


Few-shot classification:
Text: "The food was delicious but the service was slow."
Sentiment: NEUTRAL
Confidence: HIGH

This text contains both positive ("delicious") and negative ("slow") elements that balance each other out, resulting in a mixed but overall neutral sentiment.


### Chain of Thought Prompting

Encouraging Claude to "think step by step" improves reasoning quality.


In [29]:
# Chain of thought prompting
problem = """
A train leaves Station A at 9:00 AM traveling at 60 mph toward Station B.
Another train leaves Station B at 10:00 AM traveling at 80 mph toward Station A.
If the stations are 280 miles apart, at what time will the trains meet?

Think through this step by step before giving your final answer.
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    messages=[{"role": "user", "content": problem}]
)

show_response(response)

Chain of thought reasoning:


I need to find when two trains traveling toward each other will meet.

Let me set up the problem step by step.

**Given information:**
- Train A leaves at 9:00 AM at 60 mph
- Train B leaves at 10:00 AM at 80 mph
- Distance between stations: 280 miles

**Step 1: Account for the head start**
Train A gets a 1-hour head start (9:00 AM vs 10:00 AM).
In that first hour, Train A travels: 60 mph × 1 hour = 60 miles

**Step 2: Find the remaining distance when both trains are moving**
After 10:00 AM, the remaining distance between the trains is:
280 - 60 = 220 miles

**Step 3: Calculate their combined closing speed**
Once both trains are moving (after 10:00 AM), they approach each other at:
60 mph + 80 mph = 140 mph combined

**Step 4: Calculate time to meet**
Time for trains to meet = Remaining distance ÷ Combined speed
Time = 220 miles ÷ 140 mph = 22/14 hours = 11/7 hours

Converting to hours and minutes:
11/7 hours = 1 4/7 hours = 1 hour and 4/7 × 60 minutes
4/7 × 60 = 240/7 ≈ 34.3 minutes ≈ 34 minutes and 18 seconds

**Step 5: Find the meeting time**
The trains meet 1 hour and 34 minutes after 10:00 AM
10:00 AM + 1 hour 34 minutes = 11:34 AM

**Verification:**
- Train A travels from 9:00 AM to 11:34 AM = 2 hours 34 minutes = 2 34/60 hours = 2 17/30 hours
- Distance covered by Train A: 60 × (2 + 17/30) = 60 × 77/30 = 154 miles

---
## 3. Multi-Step Prompts

### Conversation History

Claude can maintain context across multiple exchanges.


In [30]:
# Multi-turn conversation
conversation = [
    {"role": "user", "content": "I'm learning Python. What should I start with?"},
    {"role": "assistant", "content": "Great choice! I recommend starting with:\n1. Variables and data types\n2. Control flow (if/else, loops)\n3. Functions\n4. Lists and dictionaries\n\nWould you like me to explain any of these in detail?"},
    {"role": "user", "content": "Yes, explain functions please."},
    {"role": "assistant", "content": "Functions are reusable blocks of code. Here's a simple example:\n\n```python\ndef greet(name):\n    return f'Hello, {name}!'\n\nresult = greet('Alice')  # Returns 'Hello, Alice!'\n```\n\nKey concepts:\n- `def` keyword defines a function\n- Parameters go in parentheses\n- `return` sends back a value\n\nWould you like to see more examples?"},
    {"role": "user", "content": "How do I use default parameters?"}
]

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=conversation
)

print("Continuing conversation (Claude remembers context):")
show_response(response)


Continuing conversation (Claude remembers context):


Default parameters let you provide fallback values when arguments aren't passed:

```python
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

# Using default parameter
print(greet("Alice"))        # "Hello, Alice!"

# Overriding default
print(greet("Bob", "Hi"))    # "Hi, Bob!"
```

Key points:
- Default parameters go after regular parameters
- You can have multiple defaults: `def func(a, b="default1", c="default2"):`
- Only the required parameters need to be provided

Want to see more advanced function features?

### Building a Simple Chatbot


In [40]:
class SimpleChatbot:
    """A simple chatbot that maintains conversation history."""
    
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.client = anthropic.Anthropic()
        self.system_prompt = system_prompt
        self.conversation_history = []
    
    def chat(self, user_message):
        """Send a message and get a response."""
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        # Get response from Claude
        response = self.client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=300,
            system=self.system_prompt,
            messages=self.conversation_history
        )
        
        # Extract and store assistant response
        assistant_message = response.content[0].text
        self.conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })
        
        return assistant_message
    
    def reset(self):
        """Clear conversation history."""
        self.conversation_history = []

# Demo the chatbot
bot = SimpleChatbot(system_prompt="You are a Python tutor. Be concise and helpful.")

print(" Chatbot Demo")
print("-" * 40)

exchanges = [
    "What is a list comprehension?",
    "Show me an example",
    "How is that different from a regular loop?"
]

for msg in exchanges:
    print(f" User: {msg}")
    response = bot.chat(msg)
    print(f" Bot: {response}")
    print("-" * 40)


 Chatbot Demo
----------------------------------------
 User: What is a list comprehension?
 Bot: A list comprehension is a concise way to create lists in Python using a single line of code.

**Basic syntax:**
```python
[expression for item in iterable]
```

**Examples:**

1. **Basic example:**
```python
# Traditional way
squares = []
for x in range(5):
    squares.append(x**2)

# List comprehension
squares = [x**2 for x in range(5)]
# Result: [0, 1, 4, 9, 16]
```

2. **With conditions:**
```python
# Only even squares
even_squares = [x**2 for x in range(10) if x % 2 == 0]
# Result: [0, 4, 16, 36, 64]
```

3. **Processing strings:**
```python
words = ['hello', 'world', 'python']
upper_words = [word.upper() for word in words]
# Result: ['HELLO', 'WORLD', 'PYTHON']
```

**Benefits:**
- More readable and concise
- Often faster than traditional loops
- Pythonic way to create lists

List comprehensions are great for simple transformations, but use regular loops for complex logic.
-----------

### Multi-Step Task Processing

Breaking complex tasks into steps and processing them sequentially.


In [32]:
# Multi-step processing: Analyze, then act
text = """
Our Q3 sales were $2.5M, up 15% from Q2. The marketing team launched 
three new campaigns that drove significant website traffic. Customer 
satisfaction scores improved to 4.5/5. However, we noticed increased 
support ticket volume, suggesting some product issues need attention.
"""

# Step 1: Extract key metrics
step1_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[{
        "role": "user",
        "content": f"""
Extract all numerical metrics from this text. Return as a bullet list.

<text>
{text}
</text>
"""
    }]
)

metrics = step1_response.content[0].text
print("Step 1 - Extracted Metrics:")
print(metrics)
print("\n" + "="*50 + "\n")

# Step 2: Generate insights based on extracted metrics
step2_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": f"""
Based on these metrics from a quarterly report:

{metrics}

Provide 3 actionable insights for the leadership team.
"""
    }]
)

print("Step 2 - Generated Insights:")
show_response(step2_response)


Step 1 - Extracted Metrics:
Here are the numerical metrics extracted from the text:

• $2.5M (Q3 sales)
• 15% (increase from Q2)
• 3 (new marketing campaigns)
• 4.5/5 (customer satisfaction score)


Step 2 - Generated Insights:


Based on these quarterly metrics, here are 3 actionable insights for the leadership team:

## 1. Scale High-Performing Marketing Strategy
With 3 new campaigns driving a strong 15% quarter-over-quarter growth, **identify which specific campaigns generated the highest ROI and allocate more budget to replicate these successful tactics in Q4**. Consider expanding the top-performing campaign to new market segments or channels.

## 2. Leverage Customer Satisfaction for Revenue Growth
Your exceptional 4.5/5 customer satisfaction score is a significant competitive advantage. **Implement a formal customer referral program and case study initiative to convert this satisfaction into measurable revenue growth**. Satisfied customers can become your most cost-effective sales channel.

## 3. Accelerate Growth Momentum with Strategic Investment
The 15% growth trajectory combined with high satisfaction suggests strong market fit. **Evaluate increasing marketing spend by 20-30% for Q4 to capitalize on this momentum**, while establishing systems to maintain service quality as you scale. Consider this an optimal time to gain market share before competitors catch up.

Each insight provides a clear path to build on your current success while addressing scalability and sustainable growth.

---
## 4. Document Summarization

Claude excels at summarizing and extracting information from documents.


In [33]:
# Sample long document (article about climate change)
long_document = """
Climate Change and Its Global Impact: A Comprehensive Overview

Introduction:
Climate change represents one of the most pressing challenges facing humanity in the 21st century. 
The phenomenon, driven primarily by human activities, has far-reaching consequences for ecosystems, 
economies, and societies worldwide. This article examines the causes, effects, and potential 
solutions to this global crisis.

Causes of Climate Change:
The primary driver of modern climate change is the emission of greenhouse gases, particularly 
carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O). These emissions result from burning 
fossil fuels for energy, deforestation, industrial processes, and agricultural practices. Since 
the Industrial Revolution, atmospheric CO2 levels have increased by over 50%, from approximately 
280 parts per million (ppm) to over 420 ppm today.

Environmental Impacts:
The consequences of climate change are already visible across the globe. Average global 
temperatures have risen by approximately 1.1°C since pre-industrial times. This warming has led 
to melting ice caps and glaciers, rising sea levels, more frequent and intense extreme weather 
events, shifts in ecosystems and biodiversity, and altered precipitation patterns affecting 
agriculture.

Economic Implications:
The economic costs of climate change are substantial and growing. The World Bank estimates that 
climate change could push an additional 132 million people into extreme poverty by 2030. 
Industries such as agriculture, insurance, and tourism face significant disruptions. However, 
the transition to clean energy also presents economic opportunities, with renewable energy 
sectors experiencing rapid growth.

Solutions and Mitigation:
Addressing climate change requires a multi-faceted approach. Key strategies include transitioning 
to renewable energy sources, improving energy efficiency, protecting and restoring forests, 
developing sustainable agriculture practices, and investing in carbon capture technologies. 
International agreements like the Paris Agreement aim to limit global warming to 1.5°C above 
pre-industrial levels.

Conclusion:
Climate change poses an existential threat that demands immediate and sustained action. While 
the challenges are immense, solutions exist and are increasingly economically viable. Success 
requires coordinated efforts from governments, businesses, and individuals worldwide.
"""

print(f"Document length: {len(long_document)} characters")
print(f"Approximate word count: {len(long_document.split())} words")


Document length: 2437 characters
Approximate word count: 319 words


In [34]:
# Basic summarization
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": f"""
Summarize the following document in 3-4 sentences, capturing the main points.

<document>
{long_document}
</document>
"""
    }]
)

show_response(response)

📝 Basic Summary:


Climate change, driven primarily by greenhouse gas emissions from human activities like fossil fuel burning and deforestation, has become one of the most critical challenges of the 21st century. The effects are already evident globally, with temperatures rising 1.1°C since pre-industrial times, leading to melting ice caps, rising sea levels, extreme weather events, and ecosystem disruptions. The economic impact is severe, with the World Bank projecting that climate change could push 132 million additional people into extreme poverty by 2030, while also disrupting key industries like agriculture and tourism. However, solutions exist through renewable energy transitions, forest protection, sustainable agriculture, and carbon capture technologies, requiring coordinated global action to limit warming to 1.5°C above pre-industrial levels as outlined in international agreements like the Paris Agreement.

In [35]:
# Structured summarization
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=400,
    messages=[{
        "role": "user",
        "content": f"""
Analyze this document and provide a structured summary with:

1. **Main Topic**: One sentence description
2. **Key Statistics**: Any numbers or data mentioned
3. **Main Arguments**: 3 bullet points
4. **Conclusion**: The document's final message

<document>
{long_document}
</document>
"""
    }]
)

print("📊 Structured Summary:")
show_response(response)


📊 Structured Summary:


# Document Analysis: Climate Change and Its Global Impact

## 1. Main Topic
This document provides a comprehensive overview of climate change as a critical global challenge, examining its human-driven causes, environmental and economic impacts, and potential solutions.

## 2. Key Statistics
- **CO2 levels**: Increased over 50% since Industrial Revolution (from ~280 ppm to 420+ ppm)
- **Global temperature rise**: Approximately 1.1°C above pre-industrial levels
- **Poverty projection**: Additional 132 million people could be pushed into extreme poverty by 2030
- **Temperature target**: Paris Agreement aims to limit warming to 1.5°C above pre-industrial levels

## 3. Main Arguments
• **Human activities are the primary driver**: Climate change is caused mainly by greenhouse gas emissions from fossil fuel burning, deforestation, industrial processes, and agriculture

• **Impacts are already widespread and costly**: Environmental consequences include rising temperatures, melting ice, sea level rise, and extreme weather, while economic costs threaten to increase global poverty significantly

• **Solutions exist but require coordinated action**: Mitigation strategies like renewable energy transition, energy efficiency, forest protection, and carbon capture are available and increasingly viable, but need global cooperation

## 4. Conclusion
Climate change represents an existential threat requiring immediate, sustained global action, but solutions are available and becoming more economically feasible, necessitating coordinated efforts from governments, businesses, and individuals worldwide.

In [36]:
# Audience-specific summarization
audiences = [
    ("Executive", "Summarize for a busy CEO. Focus on business impact and action items. 2-3 sentences max."),
    ("Technical", "Summarize for a climate scientist. Include specific data and technical details."),
    ("General Public", "Summarize for a newspaper article. Use simple language, be engaging.")
]

print("👥 Audience-Specific Summaries:\n")

for audience, instruction in audiences:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"""
{instruction}

<document>
{long_document}
</document>
"""
        }]
    )
    
    print(f"📌 {audience} Version:")
    print(response.content[0].text)
    print("\n" + "-"*50 + "\n")


👥 Audience-Specific Summaries:

📌 Executive Version:
**Business Impact:** Climate change threatens to push 132 million more people into poverty by 2030 and significantly disrupt agriculture, insurance, and tourism industries, while creating major growth opportunities in renewable energy sectors. **Action Items:** Assess your company's climate risk exposure across operations and supply chains, and evaluate investment opportunities in clean energy transition technologies to capitalize on this rapidly growing market.

--------------------------------------------------

📌 Technical Version:
## Climate Change Summary for Climate Scientists

### Key Quantitative Data:
- **Atmospheric CO2**: Increased >50% since Industrial Revolution (280 ppm → >420 ppm current)
- **Global temperature anomaly**: +1.1°C above pre-industrial baseline
- **Socioeconomic projection**: 132 million additional people in extreme poverty by 2030 (World Bank estimate)

### Primary Forcing Agents:
The document identifies

## Image Models (Vision)

Claude can analyze images through its vision capabilities. This is useful for:
- Image description and analysis
- Reading text from images (OCR)
- Answering questions about visual content
- Comparing multiple images

We'll use a publicly available image. Supported Image Formats are JPEG, PNG, GIF, WebP

In [37]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
display(Image(url=image_url, width=300))

In [38]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "url",
                        "url": image_url
                    }
                },
                {
                    "type": "text",
                    "text": "Describe this image in detail. What do you see?"
                }
            ]
        }
    ]
)

In [39]:
print("Image Analysis:")
show_response(response)

Image Analysis:


This image shows a beautiful orange tabby cat with striking golden-yellow eyes. The cat has a warm, ginger-colored coat with subtle striping patterns typical of tabby cats. Its face is the main focus of the photo, showing alert, pointed ears and prominent white whiskers extending from both sides of its face. 

The cat appears to be sitting and looking directly at the camera with an attentive, curious expression. The lighting creates a lovely warm tone that complements the cat's coloring. In the blurred background, you can see what appears to be a tiled floor and some red linear elements that are out of focus, possibly furniture legs or structural elements. The shallow depth of field keeps the cat as the clear subject while creating an attractive bokeh effect in the background.

The overall composition captures the cat's gentle, engaging personality and showcases the beautiful amber tones of both its fur and eyes.

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>